# 16 · im2col 卷积与池化

> **本节属于 Part 6 · 卷积神经网络 (CNN)。**

上一节的朴素卷积清晰但太慢。本节用经典的 **im2col** 技巧把卷积**转化为矩阵乘法**——既快，又能直接复用我们早已验证过的 `matmul` 自动求导。唯一需要自定义反向的只有 im2col 这一步（它的反向叫 col2im）。

## 学习目标

- 理解 **im2col**：把所有卷积窗口展平成列，卷积 = 一次矩阵乘法
- 实现高效的 `Conv2d`、`MaxPool2d`、`Flatten`
- 用 gradcheck 验证反向，与 PyTorch 对照，并与朴素版**对拍 + 测速**

## im2col 的思想

卷积的每个输出位置都是"一个图像块 ⋅ 卷积核"。如果把**所有图像块**各自展平成一列、堆成一个大矩阵 `cols`（形状 `(C·kh·kw, L)`，L 是输出位置数），再把卷积核展平成 `(out_ch, C·kh·kw)`，那么

$$\text{所有输出} = \text{核矩阵} \times \text{cols}$$

一次矩阵乘法就算完了整张特征图！看 `unfold`(im2col) 与 `Conv2d.forward` 的真实实现：

In [ ]:
import inspect
import time
import numpy as np
from minitorch import Tensor, nn, rel_error
from minitorch.nn.conv import unfold

print(inspect.getsource(nn.Conv2d.forward))

注意 `forward` 里：`unfold` 之后就是 `w2 @ cols`——**复用了 matmul 的 autograd**。所以我们只需为 `unfold` 写一个反向（col2im，把梯度散射回原图），其余反向全自动。

## 验证一：与朴素卷积对拍 + 测速

In [ ]:
def conv2d_naive(x, w, b, stride=1, pad=0):
    N, C, H, W = x.shape; O, _, kh, kw = w.shape
    xpad = np.pad(x, ((0,0),(0,0),(pad,pad),(pad,pad)))
    oh = (H+2*pad-kh)//stride+1; ow = (W+2*pad-kw)//stride+1
    out = np.zeros((N, O, oh, ow))
    for n in range(N):
        for o in range(O):
            for i in range(oh):
                for j in range(ow):
                    out[n,o,i,j] = np.sum(xpad[n,:,i*stride:i*stride+kh, j*stride:j*stride+kw]*w[o])+b[o]
    return out

np.random.seed(0)
x = np.random.randn(4, 3, 16, 16)
conv = nn.Conv2d(3, 8, 3, padding=1)

t0 = time.time(); naive = conv2d_naive(x, conv.weight.data, conv.bias.data, pad=1); t_naive = time.time()-t0
t0 = time.time(); fast = conv(Tensor(x)).data; t_fast = time.time()-t0
print(f"im2col vs 朴素 结果相对误差: {rel_error(fast, naive):.2e}")
print(f"朴素耗时 {t_naive*1000:.1f} ms   im2col 耗时 {t_fast*1000:.1f} ms   加速 ~{t_naive/t_fast:.0f}x")

## 验证二：梯度检查 + PyTorch 对照

`Conv2d` 的反向（对输入和对卷积核）都通过数值梯度检查；前向与 PyTorch 完全一致。（完整测试见 `tests/test_conv.py`。）

In [ ]:
import torch
from minitorch.utils import numerical_gradient
np.random.seed(1)
x = np.random.randn(2, 3, 7, 7)
conv = nn.Conv2d(3, 4, 3, padding=1)
R = np.random.randn(2, 4, 7, 7)
tx = Tensor(x); (conv(tx) * Tensor(R)).sum().backward()
g = numerical_gradient(lambda v: float((conv(Tensor(v)).data * R).sum()), x.copy())
print("Conv2d 对输入梯度 相对误差:", rel_error(tx.grad, g))

out_t = torch.nn.functional.conv2d(torch.tensor(x), torch.tensor(conv.weight.data),
                                   torch.tensor(conv.bias.data), padding=1).numpy()
print("Conv2d 前向 vs PyTorch:", rel_error(conv(Tensor(x)).data, out_t))

## MaxPool2d：下采样

最大池化在每个小窗口里取最大值，缩小特征图、保留最强响应。它的反向很特别：**梯度只回流到当初取到最大值的那个位置**（用前向记录的 argmax 路由）。

In [ ]:
print(inspect.getsource(nn.MaxPool2d.forward))

x = np.random.randn(1, 1, 4, 4)
pool = nn.MaxPool2d(2)
tx = Tensor(x); (pool(tx)*pool(tx)).sum().backward()
g = numerical_gradient(lambda v: float((pool(Tensor(v)).data**2).sum()), x.copy())
print("MaxPool2d 梯度 相对误差:", rel_error(tx.grad, g))
print("输入 4x4 -> 池化后", pool(Tensor(x)).shape[2:])

## 📦 沉淀进 minitorch

`Conv2d / MaxPool2d / Flatten`（及内部的 `unfold`/col2im）都在 **`minitorch/nn/conv.py`**，由 `tests/test_conv.py` 守护。**关键收获**：把昂贵的算子转成 matmul，既提速又能复用已验证的 autograd。

## 小练习

1. **stride 卷积**：用 `Conv2d(1,1,3,stride=2)` 对 8×8 输入卷积，输出尺寸是多少？与公式核对。
2. **im2col 提速来源**：朴素卷积慢在哪？（提示：Python 循环 vs NumPy 向量化 + 高度优化的 matmul。）
3. **平均池化（进阶）**：仿照 `MaxPool2d` 实现 `AvgPool2d`，它的反向是把梯度**均分**回窗口里每个位置。

## 小结 & 下一站

✅ 我们用 im2col 实现了高效 `Conv2d`，与朴素版对拍一致、比它快几十倍，反向通过 gradcheck 并与 PyTorch 对齐；还实现了 `MaxPool2d` 的 argmax 路由反向。

**下一站 → `17_cnn_on_mnist`**：把这些积木搭成一个 LeNet 风格的 CNN，训练它识别 MNIST，准确率**超过**我们之前的 MLP，并可视化它学到的卷积核与特征图。